In [ ]:
import sqlite3
import pandas as pd
from google.colab import drive
drive.mount("/content/drive")
P = "/content/drive/MyDrive/Project3_Procurement/"
df = pd.read_parquet(P + "awards_clean.parquet")
assert len(df) == 4839
con = sqlite3.connect(":memory:")
df.to_sql("awards", con, index=False)
def q(sql):
    return pd.read_sql_query(sql, con)
print(sqlite3.sqlite_version)

Mounted at /content/drive
3.45.1


In [ ]:
Q1 = """
SELECT category,
  COUNT(*) AS n,
  ROUND(SUM(amount_usd)) AS spend,
  ROUND(100.0 * SUM(amount_usd) /
    (SELECT SUM(amount_usd) FROM awards), 1)
    AS pct
FROM awards
GROUP BY category
ORDER BY spend DESC;
"""
print(q(Q1))

                  category     n         spend   pct
0                    Works   777  1.327031e+09  61.9
1                    Goods  2576  4.471697e+08  20.8
2      Consultant Services  1301  3.460069e+08  16.1
3  Non-consulting Services   185  2.525060e+07   1.2


In [ ]:
# Top 10 firms by spend, without the zero-value UNKNOWN row
Q2 = """
SELECT supplier_key,
  COUNT(*) AS n,
  ROUND(SUM(amount_usd)) AS spend
FROM awards
WHERE is_individual = 0
  AND supplier_key <> 'UNKNOWN | XX'
GROUP BY supplier_key
ORDER BY spend DESC
LIMIT 10;
"""
print(q(Q2))

                                        supplier_key  n        spend
0              CHINA STATE CONSTRUCTION FINANCE | CN  2  251365793.0
1                                     SINOHYDRO | CN  2  196302752.0
2         UNITED NATIONS CHILDREN S FUND UNICEF | 1W  4  144158703.0
3                      LONGJIAN ROAD AND BRIDGE | CN  1   66048524.0
4          CHINA STATE CONSTRUCTION ENGINEERING | CN  1   57416816.0
5        JIANGXI CONSTRUCTION ENGINEERING GROUP | CN  1   46446011.0
6  SHANDONG HIGHWAY BRIDGE CONSTRUCTION GROUP BRI...  1   43315459.0
7                           HALANE CONSTRUCTION | KE  2   37243538.0
8                    ZHONGMEI ENGINEERING GROUP | CN  1   32772363.0
9        CHINA INTERNATIONAL WATER AND ELECTRIC | CN  5   27522435.0


In [ ]:
# Count only firms with positive spend (same result as excluding UNKNOWN)
Q3 = """
WITH s AS (
  SELECT supplier_key,
         SUM(amount_usd) AS spend
  FROM awards
  WHERE is_individual = 0
  GROUP BY supplier_key
  HAVING SUM(amount_usd) > 0),
r AS (
  SELECT supplier_key, spend,
    ROW_NUMBER() OVER (ORDER BY spend DESC)
      AS rnk,
    SUM(spend) OVER (ORDER BY spend DESC
      ROWS BETWEEN UNBOUNDED PRECEDING
      AND CURRENT ROW) AS cum
  FROM s),
p AS (
  SELECT rnk, supplier_key,
    100.0 * cum / (SELECT SUM(spend) FROM s)
      AS cp
  FROM r)
SELECT MIN(rnk) AS n_for_80,
  (SELECT COUNT(*) FROM s) AS n_suppliers
FROM p
WHERE cp >= 80;
"""
print(q(Q3))

   n_for_80  n_suppliers
0       180         2931


In [ ]:
# Pareto chart table: first 15 firms with cumulative %, counting only firms with positive spend
Q3b = """
WITH s AS (
  SELECT supplier_key,
         SUM(amount_usd) AS spend
  FROM awards
  WHERE is_individual = 0
  GROUP BY supplier_key
  HAVING SUM(amount_usd) > 0),
r AS (
  SELECT supplier_key, spend,
    ROW_NUMBER() OVER (ORDER BY spend DESC)
      AS rnk,
    SUM(spend) OVER (ORDER BY spend DESC
      ROWS BETWEEN UNBOUNDED PRECEDING
      AND CURRENT ROW) AS cum
  FROM s),
p AS (
  SELECT rnk, supplier_key,
    100.0 * cum / (SELECT SUM(spend) FROM s)
      AS cp
  FROM r)
SELECT rnk, supplier_key,
  ROUND(cp, 1) AS cum_pct
FROM p
WHERE rnk <= 15
ORDER BY rnk;
"""
print(q(Q3b))

    rnk                                       supplier_key  cum_pct
0     1              CHINA STATE CONSTRUCTION FINANCE | CN     11.9
1     2                                     SINOHYDRO | CN     21.1
2     3         UNITED NATIONS CHILDREN S FUND UNICEF | 1W     27.9
3     4                      LONGJIAN ROAD AND BRIDGE | CN     31.1
4     5          CHINA STATE CONSTRUCTION ENGINEERING | CN     33.8
5     6        JIANGXI CONSTRUCTION ENGINEERING GROUP | CN     36.0
6     7  SHANDONG HIGHWAY BRIDGE CONSTRUCTION GROUP BRI...     38.0
7     8                           HALANE CONSTRUCTION | KE     39.8
8     9                    ZHONGMEI ENGINEERING GROUP | CN     41.3
9    10        CHINA INTERNATIONAL WATER AND ELECTRIC | CN     42.6
10   11                                CHAJU BUILDERS | KE     43.8
11   12                   SINOTEC GTSCC JOINT VENTURE | CN     44.7
12   13                                SOLITON TELMEC | KE     45.5
13   14             THE THIRD ENGINEERING BUREAU

In [ ]:
# UN agencies carry country code 1W
print(q("""SELECT supplier_key, COUNT(*) n,
  ROUND(SUM(amount_usd)) spend
  FROM awards WHERE country = '1W'
  GROUP BY supplier_key ORDER BY spend DESC"""))

                                        supplier_key  n        spend
0         UNITED NATIONS CHILDREN S FUND UNICEF | 1W  4  144158703.0
1  UNITED NATIONS HUMAN SETTLEMENTS PROGRAMME UN ...  1    5959620.0
2  UNITED NATIONS OFFICE PROJECT SERVICES UNOPS | 1W  1      39637.0


In [ ]:
Q4 = """
SELECT category,
  ROUND(100.0 * SUM(CASE WHEN tier <> 'Direct'
    THEN amount_usd ELSE 0 END)
    / SUM(amount_usd), 1) AS any_comp_pct,
  ROUND(100.0 * SUM(CASE WHEN tier = 'Full'
    THEN amount_usd ELSE 0 END)
    / SUM(amount_usd), 1) AS full_comp_pct
FROM awards
GROUP BY category
ORDER BY full_comp_pct DESC;
"""
print(q(Q4))

                  category  any_comp_pct  full_comp_pct
0                    Works          96.1           94.1
1      Consultant Services          84.6           72.4
2  Non-consulting Services          65.8           53.2
3                    Goods          60.2           44.6


In [ ]:
# Full competition and local share by category, without the 1W agencies
Q4b = """
SELECT category,
  ROUND(100.0 * SUM(CASE WHEN tier = 'Full'
    THEN amount_usd ELSE 0 END)
    / SUM(amount_usd), 1) AS full_pct,
  ROUND(100.0 * SUM(CASE WHEN is_kenyan = 1
    THEN amount_usd ELSE 0 END)
    / SUM(amount_usd), 1) AS local_pct
FROM awards
WHERE COALESCE(country, '') <> '1W'
GROUP BY category ORDER BY category;
"""
print(q(Q4b))

                  category  full_pct  local_pct
0      Consultant Services      76.2       77.5
1                    Goods      63.4       84.8
2  Non-consulting Services      54.5       98.5
3                    Works      94.1       37.2


In [ ]:
Q5 = """
SELECT category,
  ROUND(100.0 * SUM(CASE WHEN is_kenyan = 1
    THEN amount_usd ELSE 0 END)
    / SUM(amount_usd), 1) AS local_pct,
  ROUND(100.0 * SUM(CASE WHEN country IS NULL
    THEN amount_usd ELSE 0 END)
    / SUM(amount_usd), 1) AS unknown_pct
FROM awards
GROUP BY category
ORDER BY local_pct DESC;
"""
print(q(Q5))

                  category  local_pct  unknown_pct
0  Non-consulting Services       96.3          0.0
1      Consultant Services       73.7          0.5
2                    Goods       59.7          2.3
3                    Works       37.2          0.0


In [ ]:
Q6 = """
SELECT fiscal_year,
  COUNT(*) AS n_awards,
  ROUND(SUM(amount_usd)) AS spend,
  ROUND(100.0 * SUM(CASE WHEN tier = 'Direct'
    THEN amount_usd ELSE 0 END) / SUM(amount_usd), 1) AS direct_pct
FROM awards
GROUP BY fiscal_year
ORDER BY fiscal_year;
"""
print(q(Q6))

   fiscal_year  n_awards        spend  direct_pct
0         2020       730  417248124.0        10.9
1         2021      1179  192677430.0        11.1
2         2022       877  592542341.0        21.0
3         2023       680   95837863.0        14.3
4         2024       499  392654387.0         4.8
5         2025       497  297907025.0        15.0
6         2026       299  119568813.0        14.0
7         2027        78   37022643.0        16.8


In [ ]:
# SQL and pandas must agree on the top supplier, the 80% line and the supplier count
b = q(Q2)
s = (df[df["is_individual"] == 0]
     .groupby("supplier_key")["amount_usd"].sum()
     .sort_values(ascending=False))
s = s.drop("UNKNOWN | XX", errors="ignore")
assert b.loc[0, "supplier_key"] == s.index[0]
assert abs(b.loc[0, "spend"] - s.iloc[0]) < 1
cum = s.cumsum() / s.sum()
n80 = int((cum < 0.8).sum() + 1)
r3 = q(Q3)
assert int(r3.loc[0, "n_for_80"]) == n80
assert int(r3.loc[0, "n_suppliers"]) == len(s)
print("SQL and pandas agree", n80, "of", len(s), f"({n80 / len(s):.1%})")

SQL and pandas agree 180 of 2931 (6.1%)


In [ ]:
# Save all queries into one file, queries.sql, with a short title above each
import os
items = [
    ("Q1", "Spend by category"),
    ("Q2", "Top 10 suppliers (firms only, UNKNOWN excluded)"),
    ("Q3", "Pareto: how many suppliers make up 80% of firm spend"),
    ("Q3b", "Pareto chart table: first 15 suppliers with cumulative %"),
    ("Q4", "Competition share by category"),
    ("Q4b", "Full competition and local share by category, without UN agencies"),
    ("Q5", "Local share by category"),
    ("Q6", "Awards, spend and direct share by fiscal year"),
]
parts, missing = [], []
for name, title in items:
    sql = globals().get(name)
    if sql is None:
        missing.append(name)
        continue
    parts.append(f"-- {name}: {title}\n{sql.strip()}\n")
if missing:
    print("not defined yet (run those cells first):", missing)
text = ("-- Kenya World Bank-financed contract awards: SQL queries (SQLite)\n"
        "-- Table: awards (one row = one supplier's share of a contract award)\n\n"
        + "\n".join(parts))
with open(P + "queries.sql", "w") as f:
    f.write(text)
print(os.listdir(P))
print(text)

['kenya_contract_awards.csv', 'dash', 'awards_clean.parquet', 'awards_clean.csv', 'queries.sql']
-- Kenya World Bank-financed contract awards: SQL queries (SQLite)
-- Table: awards (one row = one supplier's share of a contract award)

-- Q1: Spend by category
SELECT category,
  COUNT(*) AS n,
  ROUND(SUM(amount_usd)) AS spend,
  ROUND(100.0 * SUM(amount_usd) /
    (SELECT SUM(amount_usd) FROM awards), 1)
    AS pct
FROM awards
GROUP BY category
ORDER BY spend DESC;

-- Q2: Top 10 suppliers (firms only, UNKNOWN excluded)
SELECT supplier_key,
  COUNT(*) AS n,
  ROUND(SUM(amount_usd)) AS spend
FROM awards
WHERE is_individual = 0
  AND supplier_key <> 'UNKNOWN | XX'
GROUP BY supplier_key
ORDER BY spend DESC
LIMIT 10;

-- Q3: Pareto: how many suppliers make up 80% of firm spend
WITH s AS (
  SELECT supplier_key,
         SUM(amount_usd) AS spend
  FROM awards
  WHERE is_individual = 0
  GROUP BY supplier_key
  HAVING SUM(amount_usd) > 0),
r AS (
  SELECT supplier_key, spend,
    ROW_NUMBER(